# Assignment 1: Text Classification with Naive Bayes
**Natural Language Processing — Fall 2026**

This notebook follows the assignment requirements: preprocessing, a fixed train/test split, a vocabulary built from training data only, Bag of Words, visualizations, a Naive Bayes classifier implemented from scratch, comparison with scikit-learn, and evaluation.

**Dataset expected:** `synthetic_text_data.csv` with columns `text` and `label`.

In [ ]:
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

# WordCloud is used only for the required visualizations.
try:
    from wordcloud import WordCloud
except ImportError:
    %pip -q install wordcloud
    from wordcloud import WordCloud

RANDOM_SEED = 42
TEST_SIZE = 0.2
sns.set_theme(style="whitegrid")

## 1. Load the dataset

Upload `synthetic_text_data.csv` when prompted, or place it in the repository's `data/` folder. The file is not included automatically, so the notebook checks common locations first.

In [ ]:
DATA_CANDIDATES = [
    "data/synthetic_text_data.csv",
    "synthetic_text_data.csv",
    "/content/synthetic_text_data.csv",
]

data_path = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload synthetic_text_data.csv.")
        data_path = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            "Place synthetic_text_data.csv in data/ or the notebook folder, then rerun this cell."
        )

df = pd.read_csv(data_path)
required_columns = {"text", "label"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"CSV must contain columns {required_columns}; found {list(df.columns)}")

df = df[["text", "label"]].dropna().copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(str)
df = df[df["text"].str.strip().ne("") & df["label"].str.strip().ne("")].reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Classes:", sorted(df["label"].unique()))
display(df.head())
display(df["label"].value_counts().rename("count").to_frame())

## 2. Text preprocessing

Convert text to lowercase, extract word tokens, and remove punctuation by keeping alphabetic tokens. We keep the preprocessing simple and apply the same function to every sentence.

In [ ]:
def tokenize(text):
    """Lowercase text and return alphabetic word tokens."""
    text = str(text).lower()
    return re.findall(r"[a-z]+", text)

df["tokens"] = df["text"].apply(tokenize)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))
df["sentence_length"] = df["tokens"].apply(len)

# Remove rows that become empty after preprocessing.
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)

print("Rows after preprocessing:", len(df))
display(df[["text", "clean_text", "label", "sentence_length"]].head())

## 3. Train/test split

Use a fixed random seed for reproducibility. The vocabulary and all word-based visualizations below are built from the training set only; the test set is held out for final evaluation.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print("\nTraining class distribution:")
display(train_df["label"].value_counts().rename("count").to_frame())

## 4. Data visualization

These plots use the training set to avoid using test examples for exploratory word analysis.

In [ ]:
# A. Most frequent words in training data
from collections import Counter

word_counts = Counter(token for tokens in train_df["tokens"] for token in tokens)
top_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "frequency"])

plt.figure(figsize=(12, 6))
sns.barplot(data=top_words, x="frequency", y="word")
plt.title("Top 20 Word Frequencies (Training Set)")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

# B. Class distribution in training data
class_counts = train_df["label"].value_counts().rename_axis("class").reset_index(name="count")
plt.figure(figsize=(10, 5))
sns.barplot(data=class_counts, x="class", y="count")
plt.title("Class Distribution (Training Set)")
plt.xlabel("Class")
plt.ylabel("Number of Sentences")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# C. Sentence length distribution in training data
plt.figure(figsize=(10, 5))
sns.histplot(train_df["sentence_length"], bins=20, kde=False)
plt.title("Sentence Length Distribution (Training Set)")
plt.xlabel("Sentence Length (Number of Tokens)")
plt.ylabel("Number of Sentences")
plt.tight_layout()
plt.show()

# D. One word cloud per class, from training examples only
classes = sorted(train_df["label"].unique())
fig, axes = plt.subplots(len(classes), 1, figsize=(12, max(4, 4 * len(classes))))
if len(classes) == 1:
    axes = [axes]

for ax, label in zip(axes, classes):
    class_tokens = [
        token
        for tokens in train_df.loc[train_df["label"] == label, "tokens"]
        for token in tokens
    ]
    text_for_cloud = " ".join(class_tokens)
    if text_for_cloud.strip():
        cloud = WordCloud(width=1000, height=400, background_color="white").generate(text_for_cloud)
        ax.imshow(cloud, interpolation="bilinear")
        ax.set_title(f"Word Cloud — {label}")
    else:
        ax.text(0.5, 0.5, "No words available", ha="center", va="center")
        ax.set_title(f"Word Cloud — {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Build a vocabulary and Bag of Words

The vocabulary is created **from the training tokens only**. Each sentence becomes a count vector: position *j* is the number of occurrences of vocabulary word *j* in that sentence.

In [ ]:
vocabulary = sorted({token for tokens in train_df["tokens"] for token in tokens})
word_to_index = {word: i for i, word in enumerate(vocabulary)}

def tokens_to_bow(tokens, word_to_index):
    vector = np.zeros(len(word_to_index), dtype=np.int64)
    for token in tokens:
        index = word_to_index.get(token)
        if index is not None:
            vector[index] += 1
    return vector

X_train = np.vstack([tokens_to_bow(tokens, word_to_index) for tokens in train_df["tokens"]])
X_test = np.vstack([tokens_to_bow(tokens, word_to_index) for tokens in test_df["tokens"]])
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("Vocabulary size:", len(vocabulary))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Vocabulary built from training data only:", True)
display(pd.DataFrame(X_train[:5], columns=vocabulary).iloc[:, :min(10, len(vocabulary))])

## 6. Naive Bayes implemented from scratch

This is a Multinomial Naive Bayes classifier implemented using NumPy and Python—not scikit-learn's Naive Bayes. Laplace smoothing (`alpha=1`) prevents zero word probabilities. Log probabilities are used for numerical stability.

In [ ]:
class ManualMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape
        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}

        for class_label in self.classes_:
            X_class = X[y == class_label]
            class_count = X_class.shape[0]
            self.class_log_prior_[class_label] = np.log(class_count / n_samples)

            # Sum word counts for this class.
            word_counts = X_class.sum(axis=0).astype(float)
            smoothed_counts = word_counts + self.alpha
            denominator = word_counts.sum() + self.alpha * n_features
            self.feature_log_prob_[class_label] = np.log(smoothed_counts / denominator)

        return self

    def predict(self, X):
        predictions = []
        for row in X:
            class_scores = []
            for class_label in self.classes_:
                score = self.class_log_prior_[class_label]
                score += np.dot(row, self.feature_log_prob_[class_label])
                class_scores.append(score)
            predictions.append(self.classes_[int(np.argmax(class_scores))])
        return np.asarray(predictions)

manual_model = ManualMultinomialNB(alpha=1.0)
manual_model.fit(X_train, y_train)

manual_train_pred = manual_model.predict(X_train)
manual_test_pred = manual_model.predict(X_test)

print("Manual model trained.")
print("Example test predictions:", manual_test_pred[:10])

## 7. Train scikit-learn model and compare predictions

Use `MultinomialNB` with the same Bag of Words matrices and the same `alpha=1.0` smoothing setting.

In [ ]:
sklearn_model = MultinomialNB(alpha=1.0)
sklearn_model.fit(X_train, y_train)

sklearn_train_pred = sklearn_model.predict(X_train)
sklearn_test_pred = sklearn_model.predict(X_test)

test_predictions_match = np.mean(manual_test_pred == sklearn_test_pred)
train_predictions_match = np.mean(manual_train_pred == sklearn_train_pred)

print(f"Train predictions matching: {train_predictions_match:.2%}")
print(f"Test predictions matching:  {test_predictions_match:.2%}")

mismatches = test_df.loc[manual_test_pred != sklearn_test_pred, ["text", "label"]].copy()
mismatches["manual_prediction"] = manual_test_pred[manual_test_pred != sklearn_test_pred]
mismatches["sklearn_prediction"] = sklearn_test_pred[manual_test_pred != sklearn_test_pred]

if mismatches.empty:
    print("The two models made identical predictions on the test set.")
else:
    print("Examples where predictions differ:")
    display(mismatches.head(10))
    print(
        "Possible causes include implementation details, class-prior handling, "
        "feature ordering, or numerical tie-breaking. Check these examples if differences occur."
    )

## 8. Evaluation

Report train and test accuracy for both models, then show a test-set confusion matrix as a heatmap with counts and precision, recall, and F1-score for each class.

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Manual Naive Bayes",
        "Train Accuracy": accuracy_score(y_train, manual_train_pred),
        "Test Accuracy": accuracy_score(y_test, manual_test_pred),
    },
    {
        "Model": "scikit-learn MultinomialNB",
        "Train Accuracy": accuracy_score(y_train, sklearn_train_pred),
        "Test Accuracy": accuracy_score(y_test, sklearn_test_pred),
    },
])

display(results.style.format({
    "Train Accuracy": "{:.4f}",
    "Test Accuracy": "{:.4f}",
}))

In [ ]:
labels = sorted(np.unique(np.concatenate([y_test, manual_test_pred, sklearn_test_pred])))

# Confusion matrix for the manual model
cm_manual = confusion_matrix(y_test, manual_test_pred, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_manual, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — Manual Naive Bayes (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# Confusion matrix for scikit-learn
cm_sklearn = confusion_matrix(y_test, sklearn_test_pred, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_sklearn, annot=True, fmt="d", cmap="Greens",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — scikit-learn MultinomialNB (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
print("Manual Naive Bayes — per-class metrics (test set)")
print(classification_report(y_test, manual_test_pred, labels=labels, zero_division=0))

print("scikit-learn MultinomialNB — per-class metrics (test set)")
print(classification_report(y_test, sklearn_test_pred, labels=labels, zero_division=0))

## 9. Conclusion

After running the notebook, use the actual printed metrics to write a short conclusion. Compare the test accuracy of both models, whether their predictions match, and which classes are easiest or hardest to classify based on the per-class F1-scores.

Do not invent or fill in results before running the notebook with the actual dataset.